# 🧠 Neural Network from Scratch
## Logistic Regression + Binary Cross-Entropy Loss + Gradient Descent
### Based on *The Hundred-Page Machine Learning Book* — Andriy Burkov

---

**Goal:** Predict whether a customer will make a purchase (label = 0 or 1)  
**Features:** Age (years) and Salary (USD thousands)  
**Architecture:** 1 linear layer + sigmoid activation = logistic regression  

This notebook is structured so you can **follow along on paper** at each step.


---
## 📦 0. Imports & Setup

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
print('✅ All imports successful!')

---
## 📊 1. The Dataset (12 customers)

### 📝 Paper layout — what the matrix looks like

```
Design matrix X  (12 × 2)          Labels y (12 × 1)
┌────────────────────────┐          ┌─────┐
│  age  │  salary (USDk) │          │  y  │
├───────┼────────────────┤          ├─────┤
│  22   │      25        │          │  0  │  ← unlikely to purchase
│  25   │      35        │          │  0  │
│  47   │      80        │          │  1  │  ← likely to purchase
│  52   │      95        │          │  1  │
│  46   │      82        │          │  1  │
│  56   │      90        │          │  1  │
│  23   │      27        │          │  0  │
│  30   │      50        │          │  1  │
│  40   │      60        │          │  1  │
│  39   │      47        │          │  0  │
│  53   │      95        │          │  1  │
│  48   │      88        │          │  1  │
└───────┴────────────────┘          └─────┘
   x₁         x₂                     y
```

- **Row** = one customer observation
- **Column** = one feature
- X has shape **(12, 2)**, y has shape **(12, 1)**

In [ ]:
inputs = torch.tensor([
    [22, 25], [25, 35], [47, 80], [52, 95], [46, 82], [56, 90],
    [23, 27], [30, 50], [40, 60], [39, 47], [53, 95], [48, 88]
], dtype=torch.float32)

labels = torch.tensor([
    [0], [0], [1], [1], [1], [1],
    [0], [1], [1], [0], [1], [1]
], dtype=torch.float32)

print(f'X shape : {inputs.shape}  → (n_samples=12, n_features=2)')
print(f'y shape : {labels.shape}  → (n_samples=12, 1)')
print()
print('X (age, salary):'); print(inputs.numpy())
print()
print('y (0=no purchase, 1=purchase):'); print(labels.numpy().T)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
X_np = inputs.numpy()
y_np = labels.numpy().ravel()

colors = ['#e74c3c' if yi == 0 else '#2ecc71' for yi in y_np]
ax.scatter(X_np[:, 0], X_np[:, 1], c=colors,
           s=120, edgecolors='white', linewidths=1.5, zorder=3)
for i, xi in enumerate(X_np):
    ax.annotate(f'  #{i+1}', xy=xi, fontsize=8, color='#555')

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Salary (USD thousands)', fontsize=12)
ax.set_title('Customer Dataset — 12 observations', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(handles=[mpatches.Patch(color='#e74c3c', label='y=0 (no purchase)'),
                   mpatches.Patch(color='#2ecc71', label='y=1 (purchase)')], loc='upper left')
plt.tight_layout(); plt.show()
print('💡 Notice: higher salary + higher age → more likely to purchase')

---
## 🏗️ 2. Neural Network Architecture

### 📝 Paper diagram

```
INPUT LAYER          COMPUTATION            OUTPUT LAYER
(2 features)                                (1 neuron)

   x₁ (age)    ──── w₁ ────┐
                            ├──→ z = w₁x₁ + w₂x₂ + b ──→ σ(z) ──→ ŷ
   x₂ (salary) ──── w₂ ────┘
                       b ──────────────────────────────────↑
```

**Parameters to learn:** w = [w₁, w₂] (shape 2×1) and bias b  
**Total learnable parameters = 3**

### Matrix form (all 12 samples at once)

```
Z = X · Wᵀ + b

(12×2) · (2×1) + (1,) = (12×1)   ← inner dims must match!

Then:  Ŷ = σ(Z)
```

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
fig.patch.set_facecolor('#fafafa'); ax.set_facecolor('#fafafa')

def draw_neuron(ax, x, y, label, color='#3498db', r=0.38, fontsize=10):
    ax.add_patch(Circle((x, y), r, color=color, zorder=3, linewidth=2, ec='white'))
    ax.text(x, y, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, label='', color='#555', lw=1.5):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        ax.text((x1+x2)/2+0.05, (y1+y2)/2+0.15, label,
                ha='center', va='bottom', fontsize=8.5, color='#c0392b', fontweight='bold')

draw_neuron(ax, 1.5, 4.0, 'x₁', color='#8e44ad')
draw_neuron(ax, 1.5, 2.0, 'x₂', color='#8e44ad')
ax.text(1.5, 4.52, 'age', ha='center', fontsize=9, color='#8e44ad')
ax.text(1.5, 2.52, 'salary', ha='center', fontsize=9, color='#8e44ad')
ax.text(1.5, 0.8, 'INPUT\nLAYER', ha='center', fontsize=9, color='#888', style='italic')

rect = mpatches.FancyBboxPatch((3.3, 2.2), 2.5, 1.6,
    boxstyle='round,pad=0.1', fc='#ecf0f1', ec='#bdc3c7', lw=1.5, zorder=2)
ax.add_patch(rect)
ax.text(4.55, 3.35, 'z = w₁x₁ + w₂x₂ + b', ha='center', va='center',
        fontsize=9.5, fontweight='bold', color='#2c3e50')
ax.text(4.55, 2.75, 'LINEAR COMBINATION', ha='center', va='center',
        fontsize=7.5, color='#7f8c8d', style='italic')
ax.text(4.55, 1.5, 'Z = X·Wᵀ + b', ha='center', fontsize=9, color='#c0392b',
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='#ffeaa7', ec='#fdcb6e'))

draw_neuron(ax, 7.2, 3.0, 'σ', color='#27ae60', r=0.42, fontsize=13)
ax.text(7.2, 3.65, 'sigmoid', ha='center', fontsize=8.5, color='#27ae60')
ax.text(7.2, 1.5, 'σ(z)=1/(1+e⁻ᶻ)', ha='center', fontsize=9, color='#27ae60',
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='#d5f5e3', ec='#27ae60'))

draw_neuron(ax, 9.0, 3.0, 'ŷ', color='#e74c3c', r=0.38, fontsize=12)
ax.text(9.0, 3.55, 'prediction', ha='center', fontsize=8.5, color='#e74c3c')
ax.text(9.0, 0.8, 'OUTPUT\nLAYER', ha='center', fontsize=9, color='#888', style='italic')

draw_neuron(ax, 1.5, 5.8, '+b', color='#f39c12', r=0.28, fontsize=8)
draw_arrow(ax, 1.88, 4.0, 3.3, 3.3, 'w₁')
draw_arrow(ax, 1.88, 2.0, 3.3, 2.8, 'w₂')
draw_arrow(ax, 1.78, 5.6, 3.3, 3.6, 'b')
draw_arrow(ax, 5.8, 3.0, 6.78, 3.0)
draw_arrow(ax, 7.62, 3.0, 8.62, 3.0)

ax.set_title('Neural Network Architecture: Logistic Regression',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout(); plt.show()

---
## 🔢 3. Sigmoid Activation Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Key properties:**
- $\sigma(0) = 0.5$ — boundary between classes
- $\sigma(z) \to 1$ as $z \to +\infty$,  $\sigma(z) \to 0$ as $z \to -\infty$
- Derivative: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ — needed for backprop

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
z = np.linspace(-8, 8, 400)
sig = 1 / (1 + np.exp(-z))
sig_d = sig * (1 - sig)

ax1.plot(z, sig, color='#27ae60', lw=2.5, label='σ(z)')
ax1.axhline(0.5, color='orange', ls='--', lw=1.2, label='σ(0)=0.5')
ax1.scatter([0],[0.5], color='orange', s=80, zorder=5)
ax1.set(xlabel='z', ylabel='σ(z)', title='Sigmoid Function', ylim=(-0.05,1.05))
ax1.legend(); ax1.grid(True, alpha=0.3)
for zv in [-4,-2,-1,0,1,2,4]:
    sv = 1/(1+np.exp(-zv))
    ax1.annotate(f'σ({zv:+d})={sv:.3f}', xy=(zv,sv), xytext=(zv+0.5,sv-0.09), fontsize=7, color='#555')

ax2.plot(z, sig_d, color='#e74c3c', lw=2.5, label="σ'(z)=σ(z)·(1-σ(z))")
ax2.scatter([0],[0.25], color='red', s=80, zorder=5, label="max at z=0: 0.25")
ax2.set(xlabel='z', ylabel="σ'(z)", title='Sigmoid Derivative (used in backprop)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Sigmoid: maps ℝ → (0,1)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('📝 Paper reference table:')
print(f'{"z":>5}  {"σ(z)":>8}')
print('-'*16)
for zv in [-4,-2,-1,0,1,2,4]:
    print(f'{zv:>5}  {1/(1+np.exp(-zv)):>8.4f}{"  ← decision boundary" if zv==0 else ""}')

---
## 📉 4. Binary Cross-Entropy Loss

$$\mathcal{L}(y, \hat{y}) = -\bigl[y \cdot \log(\hat{y}) + (1-y) \cdot \log(1-\hat{y})\bigr]$$

| y | simplifies to | behaviour |
|:---:|:---|:---|
| 1 | $-\log(\hat{y})$ | → 0 when ŷ → 1 ✅ |
| 0 | $-\log(1-\hat{y})$ | → 0 when ŷ → 0 ✅ |

### ✨ The beautiful cancellation with sigmoid

Substituting $\hat{y}=\sigma(z)$ into BCE and differentiating:

$$\frac{\partial \mathcal{L}}{\partial z} = \hat{y} - y$$

The $e$ and $\log$ cancel completely — **the gradient is just prediction minus truth!**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
yh = np.linspace(0.001, 0.999, 400)

ax1.plot(yh, -np.log(yh),   color='#2ecc71', lw=2.5, label='y=1: loss=-log(ŷ)')
ax1.plot(yh, -np.log(1-yh), color='#e74c3c', lw=2.5, label='y=0: loss=-log(1-ŷ)')
ax1.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.6)
ax1.set(ylim=(0,5), xlabel='Predicted ŷ', ylabel='Loss', title='BCE Loss per case')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.annotate('low loss\nwhen ŷ→1', xy=(0.9,0.1), xytext=(0.6,1.5),
             arrowprops=dict(arrowstyle='->', color='#27ae60'), fontsize=8, color='#27ae60')
ax1.annotate('low loss\nwhen ŷ→0', xy=(0.1,0.1), xytext=(0.25,1.5),
             arrowprops=dict(arrowstyle='->', color='#c0392b'), fontsize=8, color='#c0392b')

ax2.axis('off'); ax2.set_xlim(0,1); ax2.set_ylim(0,1)
ax2.set_title('📝 Worked examples', fontsize=12, fontweight='bold')
for j,h in enumerate(['y','ŷ','BCE Loss','Result']):
    ax2.text(0.05+j*0.24, 0.93, h, ha='left', fontsize=9.5, fontweight='bold')
ax2.axhline(0.88, 0.03, 0.97, color='#bdc3c7', lw=1)
for i,(yt,yp,res) in enumerate([(1,.9,'✅'),(1,.5,'⚠️'),(1,.1,'❌'),(0,.1,'✅'),(0,.5,'⚠️'),(0,.9,'❌')]):
    lv = -np.log(yp) if yt==1 else -np.log(1-yp)
    yr = 0.81 - i*0.12
    col = '#27ae60' if '✅' in res else '#e74c3c' if '❌' in res else '#f39c12'
    for j,v in enumerate([str(yt), f'{yp:.1f}', f'{lv:.4f}', res]):
        ax2.text(0.05+j*0.24, yr, v, ha='left', fontsize=9,
                 color='#2c3e50' if j<3 else col)

plt.suptitle('Binary Cross-Entropy Loss', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔗 5. Computation Graph (Forward & Backward Pass)

```
FORWARD:
  z = w₁x₁ + w₂x₂ + b  →  ŷ = σ(z)  →  L = BCE(ŷ, y)

BACKWARD (chain rule):
  ∂L/∂ŷ = -y/ŷ + (1-y)/(1-ŷ)
  ∂L/∂z = ŷ - y                   ← beautiful cancellation
  ∂L/∂w₁ = (1/N)Σ(ŷ-y)·x₁
  ∂L/∂w₂ = (1/N)Σ(ŷ-y)·x₂
  ∂L/∂b  = (1/N)Σ(ŷ-y)
```

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0,14); ax.set_ylim(0,7); ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

def box(ax, x, y, w, h, text, fc='#dfe6e9', ec='#636e72', fs=9, bold=False):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2,y-h/2), w, h,
        boxstyle='round,pad=0.08', fc=fc, ec=ec, lw=1.5, zorder=3))
    ax.text(x, y, text, ha='center', va='center', fontsize=fs,
            fontweight='bold' if bold else 'normal', zorder=4)

def arr(ax, x1, y1, x2, y2, label='', color='#636e72'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5), zorder=2)
    if label:
        ax.text((x1+x2)/2,(y1+y2)/2+0.15, label, ha='center', fontsize=8, color=color, style='italic')

ax.text(7,6.7,'➡  FORWARD PASS',ha='center',fontsize=12,fontweight='bold',color='#2d3436')
box(ax,1,5.2,1.4,.55,'x₁ (age)',fc='#d6eaf8',ec='#2980b9')
box(ax,1,4.0,1.4,.55,'x₂ (salary)',fc='#d6eaf8',ec='#2980b9')
box(ax,1,2.8,1.4,.55,'b (bias)',fc='#fef9e7',ec='#f39c12')
box(ax,3.5,5.5,1.2,.45,'w₁',fc='#d5f5e3',ec='#27ae60')
box(ax,3.5,4.5,1.2,.45,'w₂',fc='#d5f5e3',ec='#27ae60')
box(ax,5.2,5.3,.6,.45,'×',fc='#eaf2ff',ec='#3498db',bold=True)
box(ax,5.2,4.3,.6,.45,'×',fc='#eaf2ff',ec='#3498db',bold=True)
box(ax,7.2,4.8,1.2,.55,'Σ=z',fc='#f0e6ff',ec='#8e44ad',bold=True)
box(ax,9.4,4.8,1.3,.55,'σ(z)=ŷ',fc='#d5f5e3',ec='#1e8449',bold=True)
box(ax,11.7,4.8,1.4,.55,'BCE Loss',fc='#fadbd8',ec='#c0392b',bold=True)
box(ax,11.7,3.5,1.2,.5,'y (true)',fc='#fef9e7',ec='#e67e22')
arr(ax,1.7,5.2,4.6,5.3,'x₁'); arr(ax,1.7,4.0,4.6,4.3,'x₂')
arr(ax,4.1,5.5,4.9,5.3,'w₁'); arr(ax,4.1,4.5,4.9,4.3,'w₂')
arr(ax,5.5,5.3,6.6,4.95); arr(ax,5.5,4.3,6.6,4.7)
arr(ax,1.7,2.8,6.6,4.6,'b')
arr(ax,7.8,4.8,8.75,4.8); arr(ax,10.05,4.8,11.0,4.8)
arr(ax,11.7,3.75,11.7,4.52)

ax.text(7,2.3,'⬅  BACKWARD PASS (Gradients)',ha='center',fontsize=12,fontweight='bold',color='#c0392b')
for gx,gt in [(1.5,'∂L/∂w₁=(1/N)Σ(ŷ-y)x₁\n∂L/∂w₂=(1/N)Σ(ŷ-y)x₂\n∂L/∂b=(1/N)Σ(ŷ-y)'),
              (5.8,'∂L/∂z = ŷ - y\n(beautiful cancellation!\nσ and log cancel e)'),
              (9.5,'∂L/∂ŷ = -y/ŷ + (1-y)/(1-ŷ)'),
              (12.2,'∂L/∂L = 1')]:
    box(ax,gx,1.6,3.2,1.0,gt,fc='#fdf2e9',ec='#e67e22',fs=8)
    ax.annotate('',xy=(gx,2.1),xytext=(gx,4.4),
                arrowprops=dict(arrowstyle='->',color='#e74c3c',lw=1.5,ls='dashed'),zorder=2)

ax.set_title('Computation Graph — Forward & Backward Pass',fontsize=13,fontweight='bold',pad=8)
plt.tight_layout(); plt.show()

---
## ✍️ 6. Paper-Ready Manual Calculation (Sample #1)

Full hand calculation: age=22, salary=25, y=0  
Using the actual torch-initialised weights (seed=42) so you can verify against the model.

In [ ]:
torch.manual_seed(42)
_model_init = nn.Sequential(nn.Linear(2,1), nn.Sigmoid())
w1 = _model_init[0].weight.data[0,0].item()
w2 = _model_init[0].weight.data[0,1].item()
b  = _model_init[0].bias.data[0].item()

x1, x2, y_true = 22.0, 25.0, 0.0
print('='*65)
print('  PAPER CALCULATION — Forward Pass, Sample #1')
print('  Customer: age=22, salary=25 (USDk), label y=0')
print('='*65)
print(f'\nActual torch init weights (seed=42):')
print(f'  w₁ = {w1:.6f}')
print(f'  w₂ = {w2:.6f}')
print(f'  b  = {b:.6f}')

z = w1*x1 + w2*x2 + b
print(f'\n─── STEP 1: Linear Combination ───────────────────────────')
print(f'  z = w₁·x₁ + w₂·x₂ + b')
print(f'    = ({w1:.4f})×{x1} + ({w2:.4f})×{x2} + ({b:.4f})')
print(f'    = {w1*x1:.4f} + ({w2*x2:.4f}) + ({b:.4f})')
print(f'    = {z:.6f}')

y_hat = 1/(1+np.exp(-z))
print(f'\n─── STEP 2: Sigmoid Activation ───────────────────────────')
print(f'  ŷ = σ({z:.6f}) = 1/(1+e^{-z:.6f})')
print(f'    = 1/(1+{np.exp(-z):.6f}) = {y_hat:.6f}')

loss = -(y_true*np.log(y_hat+1e-12) + (1-y_true)*np.log(1-y_hat+1e-12))
print(f'\n─── STEP 3: BCE Loss (y=0, so L = -log(1-ŷ)) ─────────────')
print(f'  L = -log(1 - {y_hat:.6f}) = -log({1-y_hat:.6f}) = {loss:.6f}')

grad_z = y_hat - y_true
print(f'\n─── STEP 4: Gradient (beautiful cancellation) ─────────────')
print(f'  ∂L/∂z = ŷ - y = {y_hat:.6f} - {y_true:.0f} = {grad_z:.6f}')
print(f'  ∂L/∂w₁ = (ŷ-y)·x₁ = {grad_z:.6f} × {x1} = {grad_z*x1:.6f}')
print(f'  ∂L/∂w₂ = (ŷ-y)·x₂ = {grad_z:.6f} × {x2} = {grad_z*x2:.6f}')
print(f'  ∂L/∂b  = (ŷ-y)    = {grad_z:.6f}')

lr = 0.001
print(f'\n─── STEP 5: Weight Update (lr={lr}) ──────────────────────')
print(f'  w₁ ← {w1:.6f} - {lr}×{grad_z*x1:.6f} = {w1-lr*grad_z*x1:.8f}')
print(f'  w₂ ← {w2:.6f} - {lr}×{grad_z*x2:.6f} = {w2-lr*grad_z*x2:.8f}')
print(f'  b  ← {b:.6f}  - {lr}×{grad_z:.6f}  = {b-lr*grad_z:.8f}')
print('='*65)

---
## 🏔️ 7. Gradient Descent — The Core Intuition

> **Imagine the loss function as a hilly landscape.**  
> You are standing somewhere on this landscape — your position is the current weight values.  
> Your altitude is the loss. You want to reach the **lowest valley** (minimum loss).  
>
> **Strategy:** At each step, look at the slope beneath your feet (the gradient) and take one step **downhill** — in the direction of steepest descent.

$$w \leftarrow w - \alpha \cdot \frac{\partial \mathcal{L}}{\partial w}$$

where $\alpha$ (alpha) is the **learning rate** — how big a step you take.

### Why *negative* gradient?
The gradient $\frac{\partial L}{\partial w}$ points **uphill** (toward higher loss).  
We subtract it to go **downhill** (toward lower loss).

### What is a gradient, concretely?
- If $\frac{\partial L}{\partial w_1} > 0$: loss increases as w₁ increases → **decrease w₁**
- If $\frac{\partial L}{\partial w_1} < 0$: loss decreases as w₁ increases → **increase w₁**
- If $\frac{\partial L}{\partial w_1} = 0$: you are at a flat point (possibly a minimum!)

### This is **full-batch gradient descent**
At each step, we use **all 12 training examples** to compute the gradient.  
This gives an accurate but slower-per-step estimate compared to mini-batch SGD.


In [ ]:
# ── Loss surface visualisation (w1, w2 space) ────────────────────────
# Fix b=0 and sweep w1, w2 to show the loss landscape

def compute_loss_grid(W1_grid, W2_grid, b_fixed=0.0):
    X = inputs.numpy()
    y = labels.numpy()
    out = np.zeros_like(W1_grid)
    for i in range(W1_grid.shape[0]):
        for j in range(W1_grid.shape[1]):
            w1v, w2v = W1_grid[i,j], W2_grid[i,j]
            z = X @ np.array([[w1v],[w2v]]) + b_fixed
            yhat = 1/(1+np.exp(-z))
            yhat = np.clip(yhat, 1e-7, 1-1e-7)
            out[i,j] = -np.mean(y*np.log(yhat) + (1-y)*np.log(1-yhat))
    return out

w1_range = np.linspace(-0.05, 0.15, 60)
w2_range = np.linspace(-0.02, 0.08, 60)
W1G, W2G = np.meshgrid(w1_range, w2_range)
print('Computing loss surface... (this may take a moment)')
Z_surface = compute_loss_grid(W1G, W2G)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: 3D-style surface
cp = ax1.contourf(W1G, W2G, Z_surface, levels=30, cmap='RdYlGn_r', alpha=0.85)
plt.colorbar(cp, ax=ax1, label='BCE Loss')
ct = ax1.contour(W1G, W2G, Z_surface, levels=12, colors='white', alpha=0.3, linewidths=0.8)
ax1.set_xlabel('w₁  (age weight)', fontsize=11)
ax1.set_ylabel('w₂  (salary weight)', fontsize=11)
ax1.set_title('Loss Surface L(w₁, w₂) with b=0 — colour = loss height', fontsize=11, fontweight='bold')

# Mark the minimum
min_idx = np.unravel_index(np.argmin(Z_surface), Z_surface.shape)
ax1.scatter(W1G[min_idx], W2G[min_idx], s=200, c='gold', marker='*',
            zorder=6, label=f'Approx. minimum: w₁≈{W1G[min_idx]:.3f}, w₂≈{W2G[min_idx]:.3f}')
ax1.legend(loc='upper right', fontsize=8)

# Right: gradient arrows on the surface (quiver plot)
ax2.contourf(W1G, W2G, Z_surface, levels=30, cmap='RdYlGn_r', alpha=0.85)
ax2.contour(W1G, W2G, Z_surface, levels=12, colors='white', alpha=0.3, linewidths=0.8)

# Compute gradient at sparse grid points
skip = 6
X_np = inputs.numpy(); y_np = labels.numpy()
for i in range(0, W1G.shape[0], skip):
    for j in range(0, W1G.shape[1], skip):
        w1v, w2v = W1G[i,j], W2G[i,j]
        z = X_np @ np.array([[w1v],[w2v]])
        yhat = np.clip(1/(1+np.exp(-z)), 1e-7, 1-1e-7)
        err = yhat - y_np
        gw1 = np.mean(err * X_np[:,0:1])
        gw2 = np.mean(err * X_np[:,1:2])
        scale = 0.008
        ax2.annotate('', xy=(w1v - scale*gw1, w2v - scale*gw2),
                     xytext=(w1v, w2v),
                     arrowprops=dict(arrowstyle='->', color='white', lw=1.0))

ax2.scatter(W1G[min_idx], W2G[min_idx], s=200, c='gold', marker='*', zorder=6)
ax2.set_xlabel('w₁  (age weight)', fontsize=11)
ax2.set_ylabel('w₂  (salary weight)', fontsize=11)
ax2.set_title('Gradient Descent Direction (white arrows) = negative gradient at each point', fontsize=11, fontweight='bold')

plt.suptitle('The Loss Landscape — Gradient Descent finds the valley', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('💡 Each white arrow shows the direction gradient descent will move from that point.')
print('   All arrows point toward the yellow star (minimum loss).')

---
## 🖥️ 8. The PyTorch Model (LM Book snippet)

```python
model = nn.Sequential(
    nn.Linear(inputs.shape[1], 1),  # Z = X·Wᵀ + b
    nn.Sigmoid()                     # Ŷ = σ(Z)
)
optimizer = optim.SGD(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
```

### What `optimizer.zero_grad()` does and why it matters

PyTorch **accumulates** gradients by default — each call to `.backward()` **adds** to whatever gradient is already stored.  
If you forget `zero_grad()`, step 2's gradient = step 1's gradient + step 2's gradient — completely wrong!  
You must reset them to zero at the start of every step.

```
Step k:
  1. zero_grad()          ← wipe any gradient left from step k-1
  2. output = model(X)    ← forward pass
  3. loss = criterion(output, y)
  4. loss.backward()      ← compute gradients (fills .grad attributes)
  5. optimizer.step()     ← w ← w - lr·∇w
```


In [ ]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(inputs.shape[1], 1), nn.Sigmoid())
optimizer = optim.SGD(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print('📐 Model:'); print(model)
print()
print('📦 Parameters at initialisation:')
for name, p in model.named_parameters():
    print(f'  {name:30s}  shape={str(p.shape):10}  {p.data.numpy()}')
print()
print(f'W shape: {model[0].weight.shape} = (out_features=1, in_features=2)')
print(f'b shape: {model[0].bias.shape}   = (out_features=1,)')
print()
print('💡 nn.Linear(2,1) computes:  Z = X @ W.T + b')

In [ ]:
# Show predictions before any training
model.eval()
with torch.no_grad():
    preds_before = model(inputs)

print('Predictions BEFORE training:')
print(f'{"#":>3}  {"age":>4}  {"salary":>7}  {"y":>4}  {"ŷ":>8}  {"correct?"}')
print('-'*45)
for i in range(12):
    xi = inputs[i].numpy(); yi = labels[i].item(); yh = preds_before[i].item()
    ok = '✅' if (yh>0.5)==bool(yi) else '❌'
    print(f'{i+1:>3}  {xi[0]:>4.0f}  {xi[1]:>7.0f}  {yi:>4.0f}  {yh:>8.4f}  {ok}')
loss_before = criterion(preds_before, labels).item()
print(f'\nBCE Loss before training: {loss_before:.6f}')

---
## 🔍 9. Step-by-Step Loop Trace (First 5 Steps)

Before running 500 steps, let's watch **exactly** what happens step-by-step:  
old weights → gradient → new weights.  
This makes the SGD update rule concrete and mechanical.

In [ ]:
torch.manual_seed(42)
model_trace = nn.Sequential(nn.Linear(2,1), nn.Sigmoid())
opt_trace   = optim.SGD(model_trace.parameters(), lr=0.001)
crit_trace  = nn.BCELoss()

print('SGD TRACE — 5 steps')
print('='*80)

for step in range(5):
    # Read weights BEFORE update
    w1_old = model_trace[0].weight.data[0,0].item()
    w2_old = model_trace[0].weight.data[0,1].item()
    b_old  = model_trace[0].bias.data[0].item()

    # Forward + backward
    opt_trace.zero_grad()
    out  = model_trace(inputs)
    loss = crit_trace(out, labels)
    loss.backward()

    # Read gradients
    gw1 = model_trace[0].weight.grad[0,0].item()
    gw2 = model_trace[0].weight.grad[0,1].item()
    gb  = model_trace[0].bias.grad[0].item()

    # Apply update
    opt_trace.step()

    # Read weights AFTER update
    w1_new = model_trace[0].weight.data[0,0].item()
    w2_new = model_trace[0].weight.data[0,1].item()
    b_new  = model_trace[0].bias.data[0].item()

    print(f'\n── Step {step+1}  (Loss = {loss.item():.6f}) ──────────────────────────────')
    print(f'  param   │ old value   │  gradient   │  -lr×grad  │  new value')
    print(f'  ────────┼─────────────┼─────────────┼────────────┼────────────')
    for name, old, g, new in [('w₁', w1_old, gw1, w1_new),
                               ('w₂', w2_old, gw2, w2_new),
                               ('b ', b_old,  gb,  b_new)]:
        delta = -0.001 * g
        print(f'  {name}      │ {old:>+11.7f} │ {g:>+11.7f} │ {delta:>+10.7f} │ {new:>+11.7f}')

print('\n✅ Pattern: new = old - lr × gradient')
print('   If gradient > 0: weight decreases  (loss was increasing with w)')
print('   If gradient < 0: weight increases  (loss was decreasing with w)')

---
## 🏋️ 10. Full Training Loop — 500 Steps

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(inputs.shape[1], 1), nn.Sigmoid())
optimizer = optim.SGD(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

loss_history    = []
weight_history  = []
snapshots       = {}

print(f'{"Step":>6}  {"Loss":>10}  {"w₁":>10}  {"w₂":>10}  {"b":>10}')
print('-'*55)

for step in range(500):
    model.train()
    optimizer.zero_grad()
    out  = model(inputs)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()

    w1v = model[0].weight.data[0,0].item()
    w2v = model[0].weight.data[0,1].item()
    bv  = model[0].bias.data[0].item()
    loss_history.append(loss.item())
    weight_history.append([w1v, w2v, bv])

    if step in [0,1,2,9,49,99,249,499]:
        snapshots[step] = (loss.item(), w1v, w2v, bv)
        print(f'{step+1:>6}  {loss.item():>10.6f}  {w1v:>10.6f}  {w2v:>10.6f}  {bv:>10.6f}')

print('\n✅ Training complete!')

---
## 📈 11. Training Curves & Weight Evolution

In [ ]:
wh    = np.array(weight_history)
steps = np.arange(1, 501)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss
ax = axes[0]
ax.plot(steps, loss_history, color='#e74c3c', lw=2)
ax.fill_between(steps, loss_history, alpha=0.12, color='#e74c3c')
for sk, (lv, w1v, w2v, bv) in snapshots.items():
    ax.scatter(sk+1, lv, s=60, zorder=5, color='#c0392b')
    if sk in [0, 9, 99, 499]:
        ax.annotate(f'step {sk+1}\nL={lv:.3f}', xy=(sk+1, lv),
                    xytext=(sk+20, lv+0.03), fontsize=7.5,
                    arrowprops=dict(arrowstyle='->', lw=0.8))
ax.set(xlabel='Training Step', ylabel='BCE Loss',
       title='Loss Curve — decreasing with each step', xlim=(0,510))
ax.grid(True, alpha=0.3)

# Weights
ax = axes[1]
ax.plot(steps, wh[:,0], color='#3498db', lw=2, label='w₁ (age)')
ax.plot(steps, wh[:,1], color='#2ecc71', lw=2, label='w₂ (salary)')
ax.plot(steps, wh[:,2], color='#9b59b6', lw=2, ls='--', label='b (bias)')
ax.axhline(0, color='gray', ls=':', lw=1)
ax.set(xlabel='Training Step', ylabel='Parameter value', title='Weight Evolution')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('What happens during 500 SGD steps', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🗺️ 12. Gradient Descent Trajectory on the Loss Surface

This is the key visualisation that connects the abstract "downhill" intuition to the actual numbers.  
We plot every (w₁, w₂) position visited during training, on top of the loss contour map.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Reuse the loss surface computed earlier
# (recompute with tighter range around the actual trajectory)
w1_traj = wh[:,0]; w2_traj = wh[:,1]
w1_lo, w1_hi = w1_traj.min()-0.01, w1_traj.max()+0.01
w2_lo, w2_hi = w2_traj.min()-0.005, w2_traj.max()+0.005

w1r = np.linspace(w1_lo, w1_hi, 80)
w2r = np.linspace(w2_lo, w2_hi, 80)
W1G2, W2G2 = np.meshgrid(w1r, w2r)
print('Computing loss surface for trajectory plot...')
Z2 = compute_loss_grid(W1G2, W2G2, b_fixed=0.0)

for ax, title, w1_plot, w2_plot, n_steps in [
    (ax1, 'First 50 steps',  w1_traj[:50],  w2_traj[:50],  50),
    (ax2, 'All 500 steps',   w1_traj,       w2_traj,       500),
]:
    cp = ax.contourf(W1G2, W2G2, Z2, levels=25, cmap='RdYlGn_r', alpha=0.8)
    ax.contour(W1G2, W2G2, Z2, levels=12, colors='white', alpha=0.25, linewidths=0.7)

    # Trajectory path
    ax.plot(w1_plot, w2_plot, 'o-', color='dodgerblue', ms=3, lw=1.2,
            alpha=0.7, label='GD path')

    # Start / end markers
    ax.scatter(w1_plot[0],  w2_plot[0],  s=150, c='yellow', marker='>', zorder=6,
               edgecolors='black', linewidths=0.8, label='Start')
    ax.scatter(w1_plot[-1], w2_plot[-1], s=150, c='lime',   marker='*', zorder=6,
               edgecolors='black', linewidths=0.8, label='End')

    # Annotate every 10th step for the 50-step plot
    if n_steps == 50:
        for k in range(0, 50, 10):
            ax.annotate(f'step {k+1}', xy=(w1_plot[k], w2_plot[k]),
                        xytext=(w1_plot[k]+0.001, w2_plot[k]+0.001),
                        fontsize=7.5, color='white',
                        bbox=dict(boxstyle='round,pad=0.2', fc='#2c3e50', alpha=0.7))

    ax.set_xlabel('w₁ (age weight)', fontsize=11)
    ax.set_ylabel('w₂ (salary weight)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)

plt.colorbar(cp, ax=ax2, label='BCE Loss')
plt.suptitle('Gradient Descent Trajectory on the Loss Surface — blue dots = each SGD step',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print()
print('💡 Reading this plot:')
print('   • Warm colours (red/orange) = high loss = bad')
print('   • Cool colours (green) = low loss = good')
print('   • The path always moves toward lower loss — downhill!')
print('   • First few steps are largest (steep slope), later steps are small (near minimum)')

---
## ⚙️ 13. Learning Rate Experiment

The learning rate $\alpha$ controls **how big a step** you take downhill each iteration.

| Learning rate | Effect |
|:---|:---|
| Too small (e.g. 0.00001) | Learns correctly but very slowly — many steps to converge |
| Just right (e.g. 0.001) | Smooth convergence in a reasonable number of steps |
| Too large (e.g. 0.1) | Overshoots the valley — loss oscillates or diverges |

**The key intuition:** if you take steps that are too large, you jump over the minimum and land on the other side of the hill, possibly even higher than before.


In [ ]:
learning_rates = [0.00001, 0.0001, 0.001, 0.005, 0.01, 0.05]
colors_lr      = ['#8e44ad','#2980b9','#27ae60','#f39c12','#e67e22','#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax_loss = axes[0]
ax_traj = axes[1]

# Reuse loss surface
ax_traj.contourf(W1G2, W2G2, Z2, levels=25, cmap='RdYlGn_r', alpha=0.75)
ax_traj.contour(W1G2, W2G2, Z2, levels=10, colors='white', alpha=0.2, linewidths=0.6)

for lr, col in zip(learning_rates, colors_lr):
    torch.manual_seed(42)
    m = nn.Sequential(nn.Linear(2,1), nn.Sigmoid())
    o = optim.SGD(m.parameters(), lr=lr)
    c = nn.BCELoss()
    losses = []; w1s = []; w2s = []
    for step in range(500):
        o.zero_grad()
        loss = c(m(inputs), labels)
        loss.backward()
        o.step()
        lv = min(loss.item(), 5.0)   # cap for display
        losses.append(lv)
        w1s.append(m[0].weight.data[0,0].item())
        w2s.append(m[0].weight.data[0,1].item())

    ax_loss.plot(range(1,501), losses, color=col, lw=1.8,
                 label=f'lr={lr}', alpha=0.85)
    # Trajectory (first 100 steps only for clarity)
    ax_traj.plot(w1s[:100], w2s[:100], '-', color=col, lw=1.4, alpha=0.8, label=f'lr={lr}')
    ax_traj.scatter(w1s[0],   w2s[0],   s=60, c=col, marker='o', zorder=5)
    ax_traj.scatter(w1s[-1],  w2s[-1],  s=80, c=col, marker='*', zorder=6,
                    edgecolors='white', linewidths=0.8)

ax_loss.set(xlabel='Step', ylabel='BCE Loss', title='Loss curves for different learning rates',
            xlim=(0,500), ylim=(0, 1.8))
ax_loss.legend(fontsize=8); ax_loss.grid(True, alpha=0.3)
ax_loss.axhline(y=loss_history[-1], color='gray', ls='--', lw=1, alpha=0.5,
                label=f'Final loss (lr=0.001)')

ax_traj.set(xlabel='w₁', ylabel='w₂',
            title='Weight trajectories (first 100 steps) — ★ = final position')
ax_traj.legend(fontsize=7, loc='upper right')

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('📝 Key observations:')
print('  lr=0.00001 → barely moves, loss stays high after 500 steps')
print('  lr=0.001   → smooth convergence (this is the LM Book value)')
print('  lr=0.05+   → loss jumps erratically or diverges — overshooting!')

---
## ✅ 14. Final Predictions vs Ground Truth

In [ ]:
model.eval()
with torch.no_grad():
    preds_after = model(inputs)

print('Predictions AFTER 500 training steps:')
print(f'{"#":>3}  {"age":>4}  {"salary":>7}  {"y":>4}  {"ŷ":>8}  {"class":>6}  {"ok?"}')
print('-'*50)
correct = 0
for i in range(12):
    xi = inputs[i].numpy(); yi = labels[i].item(); yh = preds_after[i].item()
    pc = 1 if yh > 0.5 else 0
    ok = '✅' if pc==int(yi) else '❌'
    if pc==int(yi): correct += 1
    print(f'{i+1:>3}  {xi[0]:>4.0f}  {xi[1]:>7.0f}  {yi:>4.0f}  {yh:>8.4f}  {pc:>6}  {ok}')

loss_after = criterion(preds_after, labels).item()
print(f'\n  Loss:     {loss_after:.6f}  (was {criterion(torch.tensor([[0.5]]*12), labels).item():.6f} at random)')
print(f'  Accuracy: {correct}/12 = {correct/12*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (preds, title) in enumerate([
    (preds_before.detach().numpy(), 'BEFORE Training (step 0)'),
    (preds_after.detach().numpy(),  'AFTER Training (step 500)'),
]):
    ax = axes[ax_idx]
    xx1, xx2 = np.meshgrid(np.linspace(15,65,80), np.linspace(15,105,80))
    grid = torch.tensor(np.c_[xx1.ravel(), xx2.ravel()], dtype=torch.float32)
    with torch.no_grad():
        zz = model(grid).numpy().reshape(xx1.shape)
    ax.contourf(xx1, xx2, zz, levels=20, cmap='RdYlGn', alpha=0.35)
    ax.contour(xx1, xx2, zz, levels=[0.5], colors='black', linewidths=2, linestyles='--')

    for i in range(12):
        xi = inputs[i].numpy(); yi = labels[i].item()
        yh = preds[i].item() if preds.ndim>1 else preds[i]
        if hasattr(yh,'item'): yh=yh.item()
        pc = 1 if yh>0.5 else 0
        fc = '#2ecc71' if yi==1 else '#e74c3c'
        ec = 'black' if pc==int(yi) else 'blue'
        ax.scatter(xi[0], xi[1], c=fc, s=130, edgecolors=ec, linewidths=2.5, zorder=5)
        ax.annotate(f'#{i+1}', xy=xi, xytext=(xi[0]+0.5, xi[1]+1.5), fontsize=7.5)

    ax.set(xlabel='Age', ylabel='Salary (USDk)', title=title,
           xlim=(15,65), ylim=(15,105))
    ax.grid(True, alpha=0.25)
    ax.legend(handles=[mpatches.Patch(color='#2ecc71', label='y=1 purchase'),
                       mpatches.Patch(color='#e74c3c', label='y=0 no purchase'),
                       mpatches.Patch(edgecolor='blue', facecolor='white', label='misclassified')],
              loc='upper left', fontsize=8)

plt.suptitle('Decision Boundary: Before vs After Training', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🧮 15. Matrix Dimensions Cheatsheet

In [ ]:
W_np = model[0].weight.data.numpy()
b_np = model[0].bias.data.numpy()
X_np = inputs.numpy(); y_np = labels.numpy()

with torch.no_grad():
    Z_f   = (inputs @ model[0].weight.T + model[0].bias).numpy()
    Yhat  = model(inputs).numpy()

print('='*65)
print('  COMPLETE MATRIX/VECTOR CHEATSHEET')
print('='*65)
for name, val, shape_desc in [
    ('X  — Design matrix',    X_np[:4],      '(12×2) — first 4 rows'),
    ('W  — Weight matrix',    W_np,           '(1×2) = [[w₁,w₂]]'),
    ('b  — Bias',             b_np,           '(1,)'),
    ('Z=X@Wᵀ+b',             Z_f[:4].T,      '(12×1) — first 4'),
    ('Ŷ=σ(Z)',               Yhat[:4].T,     '(12×1) — first 4'),
    ('y  — True labels',      y_np.T,         '(12×1)'),
    ('Ŷ-y (errors)',          (Yhat-y_np).T,  '(12×1)'),
]:
    print(f'\n📌 {name:30s}  {shape_desc}')
    print(f'   {val}')

grad_W = (1/12) * X_np.T @ (Yhat - y_np)
grad_b = (1/12) * np.sum(Yhat - y_np)
print(f'\n📌 ∂L/∂W = (1/N) Xᵀ·(Ŷ-y)   shape: (2,1)')
print(f'   {grad_W}')
print(f'\n📌 ∂L/∂b = (1/N) Σ(Ŷ-y)     scalar: {grad_b:.6f}')
print('='*65)

In [ ]:
# Matrix dimension diagram
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0,14); ax.set_ylim(0,4); ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

def mbox(ax, cx, cy, w, h, label, sub, fc, ec, nr, nc):
    ax.add_patch(mpatches.FancyBboxPatch((cx-w/2,cy-h/2),w,h,
        boxstyle='round,pad=0.05',fc=fc,ec=ec,lw=2,zorder=3))
    ax.text(cx,cy+0.15,label,ha='center',va='center',fontsize=12,fontweight='bold',zorder=4)
    ax.text(cx,cy-0.28,sub,ha='center',va='center',fontsize=8,color='#636e72',zorder=4)
    ax.text(cx,cy-h/2-0.3,f'({nr}×{nc})',ha='center',fontsize=9,color=ec,fontweight='bold')

def op(ax,x,y,s): ax.text(x,y,s,ha='center',va='center',fontsize=18,color='#636e72',fontweight='bold')

ax.text(7,3.7,'Z = X · Wᵀ + b   →   Ŷ = σ(Z)',ha='center',fontsize=13,fontweight='bold')

mbox(ax, 1.3, 2.0, 1.8, 2.0, 'X', 'age,salary', '#d6eaf8','#2980b9', 12, 2)
op(ax,3.2,2.0,'·')
mbox(ax, 4.7, 2.0, 1.4, 0.7, 'Wᵀ','w₁,w₂',     '#d5f5e3','#27ae60',  2, 1)
op(ax,6.1,2.0,'+')
mbox(ax, 7.3, 2.0, 0.9, 2.0, 'b', 'broadcast',  '#fef9e7','#f39c12', 12, 1)
op(ax,8.4,2.0,'=')
mbox(ax, 9.6, 2.0, 0.9, 2.0, 'Z', 'linear',     '#f0e6ff','#8e44ad', 12, 1)
op(ax,10.7,2.0,'→σ→')
mbox(ax,12.4, 2.0, 0.9, 2.0, 'Ŷ', 'probs',      '#fadbd8','#c0392b', 12, 1)

ax.text(7,0.35,'📝 Inner dimensions must match: (12×2)·(2×1)=(12×1)',
        ha='center',fontsize=10,bbox=dict(boxstyle='round',fc='#fffde7',ec='#f9ca24',pad=0.4))
plt.tight_layout(); plt.show()

---
## 🔍 16. Manual NumPy vs PyTorch — Sanity Check

In [ ]:
Z_manual  = X_np @ W_np.T + b_np
Yhat_man  = 1/(1+np.exp(-Z_manual))
loss_man  = -np.mean(y_np*np.log(Yhat_man+1e-12)+(1-y_np)*np.log(1-Yhat_man+1e-12))

model.eval()
with torch.no_grad():
    loss_torch = criterion(model(inputs), labels).item()
    Yhat_torch = model(inputs).numpy()

print('🔍 Sanity Check: Manual NumPy vs PyTorch')
print('='*50)
print(f'  Loss (numpy): {loss_man:.8f}')
print(f'  Loss (torch): {loss_torch:.8f}')
print(f'  Match: {np.isclose(loss_man, loss_torch, rtol=1e-4)}')
print(f'  Max pred diff: {np.max(np.abs(Yhat_man - Yhat_torch)):.2e}')
print()
print('✅ nn.Sequential(nn.Linear, nn.Sigmoid) = σ(X @ Wᵀ + b)')

---
## 📝 17. Summary

```
╔══════════════════════════════════════════════════════════════════╗
║   GRADIENT DESCENT + LOGISTIC REGRESSION — COMPLETE SUMMARY     ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  FORWARD PASS (prediction):                                      ║
║    Z  = X · Wᵀ + b       (12×2)(2×1) → (12×1)                  ║
║    Ŷ  = σ(Z)             ∈ (0,1)                                ║
║    L  = -(1/N)Σ[y·log(ŷ) + (1-y)·log(1-ŷ)]                    ║
║                                                                  ║
║  BACKWARD PASS (gradients via chain rule):                       ║
║    ∂L/∂W = (1/N) Xᵀ·(Ŷ-y)   ← beautiful cancellation          ║
║    ∂L/∂b = (1/N) Σ(Ŷ-y)                                        ║
║                                                                  ║
║  GRADIENT DESCENT UPDATE:                                        ║
║    W ← W - lr · ∂L/∂W   (move downhill on loss surface)         ║
║    b ← b - lr · ∂L/∂b                                           ║
║                                                                  ║
║  LEARNING RATE RULES OF THUMB:                                   ║
║    too small → slow convergence                                  ║
║    too large → oscillation / divergence                          ║
║    just right → smooth, fast convergence                         ║
║                                                                  ║
║  PYTORCH LOOP (one step):                                        ║
║    optimizer.zero_grad()   ← MUST reset accumulated grads       ║
║    loss = criterion(model(X), y)                                 ║
║    loss.backward()         ← fills .grad on all parameters      ║
║    optimizer.step()        ← applies W ← W - lr·∇W             ║
╚══════════════════════════════════════════════════════════════════╝
```


In [ ]:
print('🎉 Notebook complete!')
print()
print('New sections added for Gradient Descent:')
print('  § 7  — GD intuition + loss surface + gradient arrow field')
print('  § 9  — Step-by-step loop trace (old weights → grad → new weights)')
print('  §12  — GD trajectory plotted ON the loss surface')
print('  §13  — Learning rate experiment (6 values compared)')
print()
print('Key Gradient Descent takeaways:')
print('  1. Gradient = slope of loss surface at current position')
print('  2. We move in the NEGATIVE gradient direction (downhill)')
print('  3. Learning rate = step size; wrong value → slow or diverge')
print('  4. zero_grad() is REQUIRED every step (PyTorch accumulates grads)')
print('  5. Full-batch GD uses ALL examples each step')